# 채널 A Bronze → Silver

- **추가:** 공통 타입으로 변환하고 오류·키 충돌을 격리합니다.
- **추가:** insert-only MERGE로 정상/격리 저장의 재전송·재시도 중복을 방지합니다.

일반 컴퓨트 DBR 16.1 이상과 Databricks Git 폴더를 사용합니다. 이 노트북은 아직 원격 실행하지 않았습니다. Bronze는 append-only 입력이어야 합니다.

## 1. 저장소와 모듈 연결

- project_root는 로컬 D: 경로가 아닌 /Workspace/.../databricks_project입니다.
- DBR 13.3 이상은 sys.path에 추가한 코드 디렉터리를 작업자에 배포합니다. 최신 파일을 Git 폴더에서 동기화합니다.

In [ ]:
import json
import sys
from pathlib import Path

dbutils.widgets.text("project_root", "")
configured_root = dbutils.widgets.get("project_root").strip()
if configured_root:
    ROOT = Path(configured_root)
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "01_contracts/data/reference/sku_mapping.json").is_file()), None)
if ROOT is None or not (ROOT / "02_pipeline/silver/channel_a.py").is_file():
    raise ValueError("project_root에 Databricks Git 폴더의 저장소 루트(/Workspace/...)를 입력하세요")

sys.path.insert(0, str(ROOT / "02_pipeline/silver"))
import importlib
import silver_pipeline as pipeline
if any(globals().get(name) is not None and globals()[name].isActive
       for name in ("query", "validation_query")):
    raise RuntimeError("실행 중인 Silver 쿼리를 먼저 중지한 뒤 모듈을 다시 불러오세요")
importlib.reload(pipeline)
print({"pipeline_module": pipeline.__file__, "table_creation": "SQL_DDL"})
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
MAPPING = json.loads((ROOT / "01_contracts/data/reference/sku_mapping.json").read_text(encoding="utf-8"))
print({"project_root": str(ROOT), "sku_mapping_count": len(MAPPING)})


## 2. 테이블·트리거·체크포인트

- 기본 availableNow는 이미 적재한 Bronze를 처리하고 종료합니다.
- live는 5초마다 새 입력을 기다립니다.

테이블 생성/쓰기와 체크포인트 Volume 쓰기 권한이 필요합니다. Silver 체크포인트는 Bronze와 별개입니다. 같은 경로의 writer를 동시에 실행하지 않습니다.

In [ ]:
dbutils.widgets.dropdown("trigger_mode", "availableNow", ["availableNow", "live"])
BRONZE_TABLE = "commerce_poc.bronze.events"
SILVER_TABLE = "commerce_poc.silver.events"
QUARANTINE_TABLE = "commerce_poc.silver.quarantine"
CHECKPOINT = "/Volumes/commerce_poc/bronze/checkpoints/bronze_to_silver_a_v1"

spark.sql("CREATE SCHEMA IF NOT EXISTS commerce_poc.silver")
pipeline.create_tables(spark, SILVER_TABLE, QUARANTINE_TABLE)
print({"bronze": BRONZE_TABLE, "silver": SILVER_TABLE,
       "quarantine": QUARANTINE_TABLE, "checkpoint": CHECKPOINT})


## 3. 처리 시작

- 기존 query가 실행 중이면 먼저 중지합니다. 체크포인트를 삭제하지 않습니다.
- 비동기로 시작하므로 다음 상태 셀에서 종료를 확인합니다.

In [ ]:
if "query" in globals() and query.isActive:
    raise RuntimeError("기존 Silver query를 먼저 중지한 뒤 다시 실행하세요")
query = pipeline.start_stream(
    spark, bronze_table=BRONZE_TABLE, silver_table=SILVER_TABLE,
    quarantine_table=QUARANTINE_TABLE, checkpoint=CHECKPOINT,
    sku_mapping=MAPPING, live=dbutils.widgets.get("trigger_mode") == "live",
)
print({"query_id": str(query.id), "active": query.isActive})


## 4. 실행 상태

- availableNow: 예외 없이 active=False가 되면 이번 입력 처리가 끝났습니다.
- live: active=True와 대기 상태가 정상일 수 있습니다. 실행 성공과 정확성은 별도로 확인합니다.

In [ ]:
print({"active": query.isActive, "status": query.status,
       "exception": str(query.exception()) if query.exception() else None})
if query.lastProgress:
    print({key: query.lastProgress.get(key) for key in ["batchId", "numInputRows", "timestamp"]})


## 5. 고정 6개 업무 이벤트 확인

- 해당 이벤트 6개, 오류 0개가 기대값입니다. test_run_id는 중복 키가 아닙니다.
- 앞선 단건 시험과 전체 시험의 첫 이벤트는 Silver에서 하나입니다.
- Silver 금액 검산이며 아직 Gold 구현 완료를 의미하지 않습니다.

- test_run_id는 공유한 6건 실행을 기본값으로 사용합니다. 다른 실행을 검증할 때 수정합니다. 이전 계약의 오류는 별도로 남을 수 있어 이번 실행 오류만 검산합니다.

In [ ]:
dbutils.widgets.text("test_run_id", "2776bae71e98416d89e7ded60ae3e264")
run_id = dbutils.widgets.get("test_run_id").strip()
expected_ids = [f"a-poc-v1-{index:02d}" for index in range(1, 7)]
actual = spark.table(SILVER_TABLE).where((F.col("channel_id") == "A") & F.col("event_id").isin(expected_ids))
actual_ids = {row.event_id for row in actual.select("event_id").collect()}
print({"rows": actual.count(), "distinct_ids": len(actual_ids),
       "missing_ids": sorted(set(expected_ids) - actual_ids)})
display(actual.select("event_id", "event_type", "order_id", "event_time", "data").orderBy("event_id"))
display(spark.table(QUARANTINE_TABLE).select("rule_id", "error_code", "event_id", "error_message").limit(30))

expected_amounts = json.loads((ROOT / "01_contracts/data/fixtures/channel_a/expected.json").read_text(encoding="utf-8"))["amounts"]
amounts = actual.agg(
    F.sum("data.order_amount").alias("created_order_amount"),
    F.sum("data.paid_amount").alias("paid_amount"),
    F.sum("data.refund_amount").alias("refund_amount"),
).first().asDict()
amounts["net_payment_amount"] = (amounts["paid_amount"] or 0) - (amounts["refund_amount"] or 0)
print({"actual": amounts, "expected": {key: expected_amounts[key] for key in amounts}})
assert actual.count() == 6 and len(actual_ids) == 6
assert all(amounts[key] == expected_amounts[key] for key in amounts)
if run_id:
    run_errors = spark.table(QUARANTINE_TABLE).where(F.get_json_object("raw_payload", "$.test_run_id") == run_id)
    print({"test_run_id": run_id, "quarantine_rows": run_errors.count()})
    assert run_errors.count() == 0


## 6. 중지·재실행

- 실시간 확인 후 STOP_STREAM=True로 중지합니다.
- 같은 체크포인트 재실행은 새 입력만 읽습니다. 입력이 없으면 멱등성 검증이 아닙니다.
- 오류·재전송·부분 저장 실패는 03_validate_silver.ipynb의 별도 테이블에서 확인합니다.

In [ ]:
STOP_STREAM = False
if STOP_STREAM and query.isActive:
    query.stop()
print({"active": query.isActive})
